# 12.10 模型有聽聲音嗎？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：形狀在圖片，音高在聲音：兩把鑰匙一起用**

替換每個模態，答案只改對應部分；保留新組合。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/joint.svg")))

**先想一想：**所有題都猜high，在balanced高低音上能多高分？

平衡高低音資料後，分別替換同題音訊；答案應按音高規則改變。只測waveform改變logits不能說明任務成功。

**把直覺寫成數學：**原音accuracy與replacement按規則變化率分開量；以未見頻率作holdout。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
frequency = torch.tensor([180.0, 220.0, 880.0, 1000.0])
truth = frequency > 500
always_high = torch.ones(4, dtype=torch.bool)
print("多數猜測基準", (truth == always_high).float().mean().item())
print("正式訓練：python scripts/train.py --task audio --train --steps 300")

**如何讀結果：**0.5只屬於這個構造的規則基準。真實音訊語言能力還需更多任務，不從單音外推。

**只改一件事：**只換held-out頻率而保持high/low邊界。
